# Silver — Orders (SCD1)
**GlobalMart Orchestration Lab**

| | |
|---|---|
| **Source** | `{catalog}.bronze.orders` |
| **Target** | `{catalog}.silver.orders` |
| **SCD Type** | SCD1 — orders are transactional, update in place |
| **Depends on** | Bronze Orders + Silver Customers (RI check) |

**DQ checks applied:**
- Standardize `status` casing using `initcap()` — `shipped` → `Shipped`, `PLACED` → `Placed`
- Cast `order_date` and `created_at` to DATE
- Referential integrity: flag orders whose `customer_id` does not exist in `silver.customers` — these are excluded from Silver

## Setup — Widgets & Constants

Widget values are overridden at runtime by Workflow job parameters.

In [ ]:
dbutils.widgets.text('catalog',       'your_catalog')
dbutils.widgets.text('source_schema', 'bronze')
dbutils.widgets.text('target_schema', 'silver')

CATALOG          = dbutils.widgets.get('catalog')
SOURCE_SCHEMA    = dbutils.widgets.get('source_schema')
TARGET_SCHEMA    = dbutils.widgets.get('target_schema')
SOURCE_TABLE     = f'{CATALOG}.{SOURCE_SCHEMA}.orders'
TABLE            = f'{CATALOG}.{TARGET_SCHEMA}.orders'
SILVER_CUSTOMERS = f'{CATALOG}.{TARGET_SCHEMA}.customers'

print(f'Source          : {SOURCE_TABLE}')
print(f'Target          : {TABLE}')
print(f'RI check against: {SILVER_CUSTOMERS}')

## Step 1 — Read from Bronze

In [ ]:
bronze_df = spark.table(SOURCE_TABLE)
print(f'Bronze rows: {bronze_df.count()}')
bronze_df.display()

## Step 2 — DQ Checks

**Status standardization:** the source sends mixed casing (`shipped`, `PLACED`, `Delivered`).
`initcap()` normalizes to title case consistently.

**Referential integrity:** orders whose `customer_id` is not in `silver.customers` cannot be
joined in Gold. They are logged here and excluded from Silver.

In [ ]:
from pyspark.sql.functions import initcap, to_date, col, when, lit

# Standardize status + cast dates
cleaned = bronze_df \
    .withColumn('status',     initcap(col('status'))) \
    .withColumn('order_date', to_date(col('order_date'))) \
    .withColumn('created_at', to_date(col('created_at'))) \
    .drop('_ingested_at', '_source_file', '_batch_id')

print('Status values after standardization:')
cleaned.groupBy('status').count().display()

In [ ]:
# Referential integrity check — customer_id must exist in silver.customers
silver_customers = spark.table(SILVER_CUSTOMERS) \
    .filter(col('is_current') == True) \
    .select('customer_id').distinct()

ri_check = cleaned.join(
    silver_customers.withColumnRenamed('customer_id', '_cust_ref'),
    cleaned.customer_id == col('_cust_ref'),
    'left'
).withColumn('_ri_flag',
    when(col('_cust_ref').isNull(), lit('UNKNOWN_CUSTOMER')).otherwise(lit(None))
).drop('_cust_ref')

violations = ri_check.filter(col('_ri_flag').isNotNull())
print(f'RI violations (unknown customer_id): {violations.count()}')
if violations.count() > 0:
    violations.select('order_id', 'customer_id', '_ri_flag').display()

valid_orders = ri_check.filter(col('_ri_flag').isNull()).drop('_ri_flag')
print(f'Valid orders for Silver: {valid_orders.count()}')

## Step 3 — Create Silver Table (first run only)

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TARGET_SCHEMA}')

spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {TABLE} (
        order_id    STRING,
        customer_id STRING,
        order_date  DATE,
        status      STRING,
        created_at  DATE
    )
    USING DELTA
''')

print(f'Table ready: {TABLE}')

## Step 4 — SCD1 MERGE

Update the row if `order_id` already exists (status may have changed), insert if it is new.
Safe to re-run — identical rows result in a no-op update.

In [ ]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, TABLE)

target.alias('t').merge(
    valid_orders.alias('s'),
    't.order_id = s.order_id'
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print('MERGE complete')

## Step 5 — Verify

In [ ]:
result = spark.table(TABLE)
print(f'Total rows in {TABLE}: {result.count()}')
print('Status distribution:')
result.groupBy('status').count().display()
result.display()